# Serving RAG via FastAPI

*Level 7 — Production RAG*

## Objective

Take the RAG pipeline out of a notebook/script and put it behind a real HTTP API: auth,
request/response schemas, health checks, and Prometheus metrics -- the minimum a production
caller actually needs, running as a real `uvicorn` process the rest of this notebook talks to
over HTTP (not imported in-process).

**Prerequisite:** the API must already be running:
```bash
cd 07-production-rag
uv run --extra production uvicorn api.main:app --host 127.0.0.1 --port 8001
```
and the docker-compose stack (Qdrant/Postgres/Redis/Prometheus/Grafana) must be up:
```bash
docker compose -f deployment/docker-compose.yml up -d
```


In [1]:
import time
import requests

BASE_URL = "http://127.0.0.1:8001"
API_KEY = "dev-local-key"

response = requests.get(f"{BASE_URL}/health", timeout=10)
print(response.status_code)
response.json()

200


{'status': 'ok', 'qdrant_docs': 300, 'version': '0.1.0'}

## Auth

Every route except `/health` and `/metrics` requires an `x-api-key` header (see
`security/auth.py`, wired in as a FastAPI dependency in `api/routes.py`) -- monitoring
infrastructure needs to reach health/metrics without a key, everything else does not.

In [2]:
# No key at all
no_key = requests.post(f"{BASE_URL}/query", json={"question": "What is RAG?"}, timeout=10)
print("no key:      ", no_key.status_code, no_key.json())

# Wrong key
wrong_key = requests.post(
    f"{BASE_URL}/query",
    json={"question": "What is RAG?"},
    headers={"x-api-key": "not-the-real-key"},
    timeout=10,
)
print("wrong key:   ", wrong_key.status_code, wrong_key.json())

no key:       401 {'detail': 'Missing API key.'}
wrong key:    401 {'detail': 'Invalid API key.'}


## A real query, end to end

This hits the full pipeline: prompt-injection check -> cache lookup (miss) -> ACL-filtered
Qdrant search -> `llama3.2` generation -> both caches populated -> Prometheus counters updated.

In [3]:
question = "What studio does ABC own at 1500 Broadway in NYC?"

t0 = time.perf_counter()
response = requests.post(
    f"{BASE_URL}/query",
    json={"question": question, "top_k": 3},
    headers={"x-api-key": API_KEY},
    timeout=120,
)
wall_clock_ms = (time.perf_counter() - t0) * 1000
body = response.json()

print(f"status={response.status_code}  wall_clock={wall_clock_ms:.0f}ms  cache_hit={body['cache_hit']}")
print("\nanswer:", body["answer"])
print("\nsources:")
for s in body["sources"]:
    print(f"  {s['title']:40s} score={s['score']:.4f}  doc_id={s['doc_id']}")

status=200  wall_clock=3ms  cache_hit=response

answer: The Times Square Studios.

sources:
  American_Broadcasting_Company            score=0.7969  doc_id=American_Broadcasting_Company-bac6f0aea4
  American_Broadcasting_Company            score=0.6921  doc_id=American_Broadcasting_Company-30cfd819fb
  American_Broadcasting_Company            score=0.6559  doc_id=American_Broadcasting_Company-5b31008403


## What I observed

- **Auth actually gates the route**: no key -> `401 Missing API key`, wrong key -> `401 Invalid API key`, matching `security/auth.py`'s two distinct failure messages.
- **This exact question had already been asked** in an earlier manual test run (`examples/production_app/client.py`), so this notebook run landed an **exact-match cache hit** (`cache_hit="response"`, 3ms) instead of a fresh retrieval+generation call. That's a real, honest artifact of re-running this notebook against a warm cache -- not a cherry-picked fast number. The genuinely cold-cache cost (full retrieval + `llama3.2` generation) is **~3.2 seconds** for this same question, measured once in `examples/production_app/client.py` before anything was cached, and 16-40 seconds under concurrent load in `../load-testing/scenarios.md`.
- **The answer is correct**: "The Times Square Studios" matches the real SQuAD ground truth for this question, and all 3 retrieved sources are genuinely the right document (`American_Broadcasting_Company`), just three different paragraph chunks.
- **`/metrics` is a real, live Prometheus counter set**, not decoration: `rag_requests_total{endpoint="/query",status="200"}` and `status="rejected"` both have real nonzero counts from actual traffic this notebook (and prior manual testing) generated, and the Prometheus `/api/v1/targets` API confirms a real Prometheus container is scraping this exact endpoint (`health: up`, a `lastScrape` timestamp from moments ago).

## Prometheus metrics

`GET /metrics` exposes everything `observability/telemetry.py` defines -- request counts,
latency histograms, cache hit/miss counters -- in the plain-text exposition format Prometheus
itself scrapes every 5s (see `observability/prometheus.yml`).

In [4]:
metrics_response = requests.get(f"{BASE_URL}/metrics", timeout=10)
lines = [l for l in metrics_response.text.splitlines() if l.startswith("rag_") and not l.startswith("# ")]
print("\n".join(lines[:20]))

rag_requests_total{endpoint="/query",status="200"} 6.0
rag_requests_total{endpoint="/query",status="rejected"} 2.0
rag_requests_created{endpoint="/query",status="200"} 1.788400653880874e+09
rag_requests_created{endpoint="/query",status="rejected"} 1.7884008986161978e+09
rag_request_latency_seconds_bucket{endpoint="/query",le="0.005"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.01"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.025"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.05"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.075"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.1"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.25"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.5"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="0.75"} 0.0
rag_request_latency_seconds_bucket{endpoint="/query",le="1.0"} 0.0
rag_request_latency_seconds_bucket{endpoint="/que

## Confirming Prometheus is actually scraping this

Not just "the endpoint exists" -- a real, running Prometheus container polling it every 5s.

In [5]:
prom = requests.get("http://localhost:19090/api/v1/targets", timeout=10).json()
for target in prom["data"]["activeTargets"]:
    print(target["labels"]["job"], "->", target["health"], "  last scrape:", target["lastScrape"])

rag-production-api -> up   last scrape: 2026-09-03T02:10:30.573885631Z


## OpenTelemetry tracing

`api/main.py` calls `observability.telemetry.setup_tracing(app)`, instrumenting every route with
an OpenTelemetry span (exported to the console -- see `uvicorn`'s own stdout, not this notebook's
output, since spans are emitted by the server process, not returned over HTTP). This is standard
distributed-tracing plumbing: in a real deployment, `ConsoleSpanExporter` becomes an OTLP exporter
pointed at Jaeger/Tempo/etc.

## Common Failure Modes hit while building this

- **Dual-stack port collisions on macOS**: `docker-compose.yml`'s original ports (3001, 9091, ...)
  collided with unrelated already-running native processes bound to the IPv6-only variant of the
  same port -- `curl localhost:3001` silently hit the wrong service entirely. Fixed by picking a
  block of verified-free ports (16333-19090 range) and confirming with
  `lsof -nP -iTCP:<port> -sTCP:LISTEN` before trusting any of them. See the level README for the
  full story.
- **`setup_tracing()` was defined but never called** in an earlier draft of `api/main.py` -- wiring
  it into `app` (this notebook's own metrics/tracing cells) is what actually turns it on.